In [1]:
import torch
from model import Generator  
from train import train, validation
#from train_profile import train
from model import Generator, Discriminator
import torch.optim as optim
from data import readFile, make_temporal_batches
import xarray as xr
import numpy as np
from einops import rearrange

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [2]:
def get_data(year):

    input = readFile(f'/scratch/jl14811/GCM_2005-2014_pr/ACCESS-CM2/pr_{year}*.nc', 'pr', 1, 21, 31)
    #input = input.reshape(-1, 1, 24, 21, 31).sum(axis=2)
    mean_val = np.nanmean(input)
    input = np.where(np.isnan(input), 0.0, input) * 3600 * 24
    print(input.shape)
    
    
    target = readFile(f'/scratch/jl14811/AORC_2011_126_186/APCP_surface_{year}*.nc', 'APCP_surface', 24, 126, 186)
    target = target.reshape(-1, 4, 6, 126, 186).sum(axis=2)
    mean_val = np.nanmean(target)
    target = np.where(np.isnan(target), 0.0, target)
    print(target.shape)
    
    
    input, target = make_temporal_batches(input, target, True)
    print(input.shape)
    print(target.shape)
    return input, target

In [3]:
model = Generator().to(device)
model.load_state_dict(torch.load('model_weights/models_aorc_aorc_24-6.pth'))
model.eval()

Generator(
  (input_bn): BatchNorm3d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (time_embed): Embedding(20, 128)
  (input_pad): ReflectionPad3d((1, 1, 1, 1, 0, 0))
  (res1): ResidualBlock3D(
    (padding_layer): ReflectionPad3d((1, 1, 1, 1, 1, 1))
    (conv1): Conv3d(1, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), bias=False)
    (norm1): InstanceNorm3d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (conv2): Conv3d(128, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), bias=False)
    (norm2): InstanceNorm3d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (relu): ReLU(inplace=True)
    (adjust_conv): Conv3d(1, 128, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (adjust_norm): InstanceNorm3d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
  )
  (res2): ResidualBlock3D(
    (padding_layer): ReflectionPad3d((1, 1, 1, 1, 1, 1))
    (conv1): Conv3d(128, 128, kernel_size=(3, 3, 3), 

In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

Total parameters: 7972355
Trainable parameters: 7972355


In [4]:
for year in range(2011, 2012, 1):
    input, target = get_data(year)
    x = torch.tensor(input, device=device).float()
    y = torch.tensor(target, device=device).float()
    print(year)
    for i in range(0, 73, 1):
        validation(model, x[i:i+1], y[i:i+1], year)

final shape (365, 1, 21, 31)
(365, 1, 21, 31)
final shape (365, 24, 126, 186)
(365, 4, 126, 186)
(73, 1, 5, 21, 31)
(73, 1, 12, 126, 186)
2011


/vast/jl14811/myenv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/vast/jl14811/myenv/lib/python3.13/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
